# PiShield — a Shield Layer for linear requirements

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihaela-stoian/PiShield/blob/hierarchical-requirements/examples/shield_layer_linear.ipynb)

This notebook runs **end-to-end with no external downloads**. It shows the Shield Layer for **linear** requirements on a **two-sided band** constraint:

$$0 \;\leq\; y_0 - y_1 \;\leq\; 0.12$$

i.e. $y_0$ must be at least $y_1$ (an ordering) *and* at most $0.12$ above it (a tolerance). The Shield Layer takes **any** prediction and returns one that satisfies the requirements — a hard guarantee, by construction, and differentiable so it can sit inside a network.

> Two-sided bands (and equalities like $y_0 = y_1$) are handled as pairs of linear inequalities. Plain monotonic chains ($y_0 \geq y_1 \geq y_2$) are just the one-sided special case.

## Setup

In [ ]:
import importlib.util, subprocess, sys, os

root = os.path.abspath(os.getcwd())
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, 'pishield')):
    root = os.path.dirname(root)
if os.path.isdir(os.path.join(root, 'pishield')):
    sys.path.insert(0, root)
    print('Using local PiShield at', root)
elif importlib.util.find_spec('pishield') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/mihaela-stoian/PiShield.git@hierarchical-requirements'],
                   check=True)
    print('Installed PiShield')
else:
    print('PiShield already available')

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from pishield.shield_layer import build_shield_layer

torch.manual_seed(0)

## 1. Write the requirements file

Each requirement is a linear inequality (`>=` or `>`). A two-sided band is simply the two inequalities that bound the gap `y_0 - y_1` from below and above. We write the upper edge as `y_1 - y_0 >= -band`, which is exactly `y_0 - y_1 <= band`.

In [ ]:
band = 0.12  # y_0 may exceed y_1, but by at most this much

requirements_path = "band_requirements.txt"
with open(requirements_path, "w") as f:
    f.write("ordering y_0 y_1\n"
            "y_0 - y_1 >= 0\n"       # lower edge: y_0 - y_1 >= 0
            f"y_1 - y_0 >= -{band}\n")  # upper edge: y_0 - y_1 <= band

print(open(requirements_path).read())

## 2. What the Shield Layer does

Before any training, look at the layer in isolation. We feed it a cloud of **arbitrary** points spread across the plane; PiShield returns, for each one, the closest point that satisfies the band. Points already inside are left untouched; points above the band are pulled down to the upper edge and points below are pushed up to the lower edge.

In [ ]:
shield = build_shield_layer(2, requirements_path)

# A cloud of *arbitrary* predictions, scattered all over the plane.
cloud = torch.rand(1500, 2) * 0.8 + 0.1
projected = shield(cloud.clone())

gap = cloud[:, 0] - cloud[:, 1]
in_band = (gap >= -1e-6) & (gap <= band + 1e-6)
pgap = projected[:, 0] - projected[:, 1]
after = int(((pgap >= -1e-6) & (pgap <= band + 1e-6)).sum())
print(f"in the band before PiShield: {int(in_band.sum())}/{len(cloud)}")
print(f"in the band after  PiShield: {after}/{len(cloud)}")

GREEN, RED = "#5a9e4a", "#d0433a"
lo, hi = 0.1, 0.9

def draw_band(ax):
    ax.fill_between([lo, hi], [lo, hi], [lo - band, hi - band], color=GREEN, alpha=0.12, lw=0, zorder=0)
    ax.plot([lo, hi], [lo, hi], "--", color="#888", lw=1, zorder=1)
    ax.plot([lo, hi], [lo - band, hi - band], "--", color="#888", lw=1, zorder=1)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.6), sharex=True, sharey=True)
draw_band(axes[0])
axes[0].scatter(cloud[in_band, 0], cloud[in_band, 1], s=10, c=GREEN, alpha=.5, edgecolors="none", zorder=2)
axes[0].scatter(cloud[~in_band, 0], cloud[~in_band, 1], s=10, c=RED, alpha=.4, edgecolors="none", zorder=2)
axes[0].set_title("Arbitrary predictions")
draw_band(axes[1])
axes[1].scatter(projected[:, 0], projected[:, 1], s=10, c=GREEN, alpha=.5, edgecolors="none", zorder=2)
axes[1].set_title("After PiShield  (all in the band)")
for ax in axes:
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
    ax.set_xlabel("$y_0$"); ax.grid(True, ls=":", alpha=.3)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[0].set_ylabel("$y_1$")
fig.suptitle("PiShield projects ANY output into the band  $0 \\leq y_0 - y_1 \\leq %.2f$" % band, y=1.01)
plt.tight_layout(); plt.show()

## 3. A learning task

Now a small regression task. Each target has a **level** $c(x)$ driven by the inputs, and the two outputs straddle it: $y_0 = c + g/2$, $y_1 = c - g/2$. The gap $g$ is drawn from a **U-shaped** distribution over $[0, \text{band}]$.

In [ ]:
from torch.distributions import Beta

N, in_dim = 1000, 4
X = torch.randn(N, in_dim)
w = torch.randn(in_dim, 1)

# A learnable "level" per point, driven by the inputs...
center = torch.sigmoid(X @ w).squeeze() * 0.5 + 0.25
# ...and a gap drawn from a U-shaped (bimodal) distribution over [0, band]:
# most points sit close to *either* edge of the band, few in the middle.
gap = Beta(0.4, 0.4).sample((N,)) * band
Y = torch.stack([center + gap / 2, center - gap / 2], dim=1)

viol = ((Y[:, 0] - Y[:, 1] < -1e-6) | (Y[:, 0] - Y[:, 1] > band + 1e-6)).sum()
print(f"targets violating the band: {int(viol)}/{N} (feasible by construction)")

fig, ax = plt.subplots(figsize=(4.8, 4.6))
draw_band(ax)
ax.scatter(Y[:, 0], Y[:, 1], s=12, c=GREEN, alpha=.45, edgecolors="white", linewidths=.25, zorder=2)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
ax.set_xlabel("$y_0$"); ax.set_ylabel("$y_1$"); ax.grid(True, ls=":", alpha=.3)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## 4. Train two networks — without and with PiShield

We train two MLPs with the same setup, **each with its own progress bar**:

- **NN (no PiShield)** — a plain network trained on MSE;
- **NN + PiShield** — the same network with the Shield Layer inside `forward`, so it is trained *through* the shield. Because the layer is differentiable, this is just ordinary backpropagation.

We record each network's band violations at every epoch. The shielded one satisfies the band on every forward pass, so its count is **0 from the very first step**.

In [ ]:
def num_violations(out):
    gap = out[:, 0] - out[:, 1]
    return int(((gap < -1e-6) | (gap > band + 1e-6)).sum())

def make_mlp():
    return nn.Sequential(nn.Linear(in_dim, 16), nn.Tanh(), nn.Linear(16, 2))

class ShieldedMLP(nn.Module):
    """The same MLP, but its output passes through the Shield Layer inside
    forward() — so the band holds on every forward pass, training included."""
    def __init__(self):
        super().__init__()
        self.model = make_mlp()
        self.shield = build_shield_layer(2, requirements_path)

    def forward(self, x):
        return self.shield(self.model(x))

def train(model, label, epochs=250, lr=5e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    hist = []                                   # band violations per epoch
    progress = tqdm(range(epochs), desc=f"training {label}")
    for _ in progress:
        opt.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        opt.step()
        with torch.no_grad():
            hist.append(num_violations(model(X)))
        progress.set_postfix(loss=f"{loss.item():.4f}", viol=hist[-1])
    return hist

baseline = make_mlp()      # trained WITHOUT PiShield
shielded = ShieldedMLP()   # trained WITH PiShield (shield is inside forward)

base_hist = train(baseline, "NN (no PiShield)")
shield_hist = train(shielded, "NN + PiShield")

with torch.no_grad():
    base_out = baseline(X)
    shield_out = shielded(X)

base_mse = ((base_out - Y) ** 2).mean().item()
shield_mse = ((shield_out - Y) ** 2).mean().item()
print(f"\nNN (no PiShield) : MSE = {base_mse:.4f},  violations = {num_violations(base_out)}/{N}")
print(f"NN + PiShield    : MSE = {shield_mse:.4f},  violations = {num_violations(shield_out)}/{N}  (guaranteed 0)")

## 5. Visualising the result

Left, the real data filling the band. Middle, the network trained **without** PiShield: it mostly stays feasible but a few outputs slip across an edge (red). Right, the network trained **with** PiShield — feasible by construction, no violations. The second plot shows the key property: the shielded network's violation count is **zero at every epoch**, while the unconstrained one only drifts towards the band as training proceeds and never actually guarantees it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6), sharex=True, sharey=True)

base = base_out.numpy(); shd = shield_out.numpy(); real = Y.numpy()
gap_b = base_out[:, 0] - base_out[:, 1]
bad = ((gap_b < -1e-6) | (gap_b > band + 1e-6)).numpy()

# 1 — real data
draw_band(axes[0])
axes[0].scatter(real[:, 0], real[:, 1], s=12, c=GREEN, alpha=.45, edgecolors="white", linewidths=.25, zorder=2)
axes[0].set_title("Real data")
# 2 — plain NN (no PiShield), violators highlighted
draw_band(axes[1])
axes[1].scatter(base[~bad, 0], base[~bad, 1], s=12, c="#3f72c4", alpha=.4, edgecolors="none", zorder=2, label="in band")
axes[1].scatter(base[bad, 0], base[bad, 1], s=22, c=RED, alpha=.8, edgecolors="white", linewidths=.3, zorder=3,
                label=f"violates ({int(bad.sum())})")
axes[1].set_title("NN (no PiShield)"); axes[1].legend(loc="upper left", fontsize=8, framealpha=.9)
# 3 — NN + PiShield (trained through the shield): feasible by construction
draw_band(axes[2])
axes[2].scatter(shd[:, 0], shd[:, 1], s=12, c=GREEN, alpha=.45, edgecolors="none", zorder=2, label="0 violations")
axes[2].set_title("NN + PiShield"); axes[2].legend(loc="upper left", fontsize=8, framealpha=.9)
for ax in axes:
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
    ax.set_xlabel("$y_0$"); ax.grid(True, ls=":", alpha=.3)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[0].set_ylabel("$y_1$")
plt.tight_layout(); plt.show()

# The shielded network never leaves the band — at *every* epoch, not just the end.
fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(base_hist, color="#3f72c4", lw=1.5, label="NN (no PiShield)")
ax.plot(shield_hist, color=GREEN, lw=2, label="NN + PiShield")
ax.set_xlabel("epoch"); ax.set_ylabel("band violations"); ax.set_ylim(bottom=-5)
ax.grid(True, ls=":", alpha=.4); ax.legend(framealpha=.9)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.set_title("Band violations during training — NN + PiShield stays at 0 throughout")
plt.tight_layout(); plt.show()

## 6. Takeaway

The Shield Layer turns a set of **linear requirements** into a hard guarantee on the output. It is:

- **general** — one-sided orderings, two-sided **bands**, and **equalities** are all just linear inequalities;
- **exact** — it returns the closest feasible point, leaving already-valid predictions unchanged, and gives **zero** violations by construction;
- **differentiable** — gradients flow through it, so it can wrap a network during training as well as correct its outputs at inference.

Whatever the network predicts, the result satisfies the requirements.